<a href="https://colab.research.google.com/github/muhammetalicvs-prog/flyrank-ml/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

The purpose of this section is not to grade the FlyRank research paper. The paper was written for a broad audience and reports its methods openly. I am using two of its findings to practise asking the same constructive validation questions that I should ask about my own work.

### Finding 1 — Model performance on identifying useful candidates

The paper reports that a model can rank or identify content records that may be useful candidates for review.

**My methodology question:**
How was the positive label defined, and which source columns were used to create it? In particular, I would want to confirm that none of the features were calculated from the same outcome window or from columns that directly contributed to the label. This would help establish that the model measured an out-of-sample pattern rather than indirectly reading the answer.

A stronger validation description would clearly show the feature window, label window, excluded label-derived columns, and the exact point in time when a prediction would have been made.

### Finding 2 — Reported model performance across the evaluated dataset

The paper reports measured model performance on its evaluation data.

**My methodology question:**
Does the validation split represent the way the model would be used on new data? For example, were records from the same client or site allowed to appear in both training and test sets, or was validation grouped by client? If the intended use is to rank records for clients that were not seen during training, a client-grouped split would provide stronger evidence than a random row split.

It would also be useful to report the outcome base rate and compare random-split results with grouped or time-aware results. The difference between those results would show how much of the measured performance may come from patterns shared within existing clients.

These questions are intended to make the findings easier to interpret and reproduce, not to suggest that the reported findings are invalid.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [2]:
!git clone https://github.com/muhammetalicvs-prog/flyrank-ml.git /content/flyrank-ml

Cloning into '/content/flyrank-ml'...
remote: Enumerating objects: 133, done.
remote: Counting objects: 100% (133/133), done.
remote: Compressing objects: 100% (104/104), done.
remote: Total 133 (delta 46), reused 78 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (133/133), 1.90 MiB | 6.65 MiB/s, done.
Resolving deltas: 100% (46/46), done.


In [4]:
from pathlib import Path

data_path = Path(
    "/content/flyrank-ml/data/raw/content_refresh_anonymized.csv"
)

print("Dosya mevcut mu:", data_path.exists())
print("Dosya yolu:", data_path)

Dosya mevcut mu: True
Dosya yolu: /content/flyrank-ml/data/raw/content_refresh_anonymized.csv


In [6]:
!rm -rf /content/flyrank-ml
!git clone https://github.com/muhammetalicvs-prog/flyrank-ml.git /content/flyrank-ml

Cloning into '/content/flyrank-ml'...
remote: Enumerating objects: 133, done.
remote: Counting objects: 100% (133/133), done.
remote: Compressing objects: 100% (104/104), done.
remote: Total 133 (delta 46), reused 78 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (133/133), 1.90 MiB | 10.55 MiB/s, done.
Resolving deltas: 100% (46/46), done.


In [8]:
print(df.columns.tolist())

for col in df.columns:
    if any(word in col.lower() for word in ["trend", "declin", "label", "target"]):
        print(f"\n{col}")
        print(df[col].value_counts(dropna=False).head(10))

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

trend_direction
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

trend_pct
trend_pct
 NaN      3388
-100.0    1395
 0.0       443
-50.0      281
-66.7    

In [10]:
# ============================================================
# 2. My model under an honest split (before/after)
# Before: random row split
# After: client-grouped split with zero client overlap
# ============================================================

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# ------------------------------------------------------------
# 1. Load data
# ------------------------------------------------------------

DATA_PATHS = [
    Path("/content/flyrank-ml/data/raw/content_refresh_anonymized.csv"),
    Path("../data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
]

data_path = next((path for path in DATA_PATHS if path.exists()), None)

if data_path is None:
    raise FileNotFoundError(
        "Dataset could not be found. Check that the repository was cloned "
        "to /content/flyrank-ml and that the CSV exists under data/raw/."
    )

df = pd.read_csv(data_path)
# Week 5 target:
# 1 = impressions trend is down
# 0 = all other observed trend categories
df["is_declining_label"] = (
    df["trend_direction"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("down")
    .astype(int)
)

print("\nCreated target: is_declining_label")
print(df["is_declining_label"].value_counts())
print(
    "Overall decline rate:",
    round(df["is_declining_label"].mean(), 4),
)

print(f"Dataset path: {data_path}")
print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")


# ------------------------------------------------------------
# 2. Resolve required columns
# ------------------------------------------------------------

TARGET_CANDIDATES = [
    "is_declining_label",
    "decline_proxy",
    "is_declining",
]

GROUP_CANDIDATES = [
    "client_id",
    "client_hash_id",
]

target_col = next(
    (column for column in TARGET_CANDIDATES if column in df.columns),
    None,
)

group_col = next(
    (column for column in GROUP_CANDIDATES if column in df.columns),
    None,
)

if target_col is None:
    raise KeyError(
        f"No target column found. Tried: {TARGET_CANDIDATES}"
    )

if group_col is None:
    raise KeyError(
        f"No client grouping column found. Tried: {GROUP_CANDIDATES}"
    )

print(f"\nTarget column: {target_col}")
print(f"Grouping column: {group_col}")


# ------------------------------------------------------------
# 3. Define a leakage-conscious feature set
# ------------------------------------------------------------

# Identifiers, labels, label-derived fields, free text, and operational
# outputs are excluded from the model feature set.
explicit_exclusions = {
    target_col,
    group_col,
    "content_id",
    "content_hash_id",
    "decline_proxy",
    "is_declining",
    "is_declining_label",
    "trend",
    "trend_pct",
    "trend_direction",
    "reason_code",
    "action_label",
    "action_score",
    "rank",
}

# Also exclude columns whose names strongly suggest target or post-outcome data.
leakage_name_terms = [
    "label",
    "target",
    "declin",
    "outcome",
    "future",
    "april",
]

automatic_exclusions = {
    column
    for column in df.columns
    if any(term in column.lower() for term in leakage_name_terms)
}

excluded_columns = explicit_exclusions.union(automatic_exclusions)

candidate_features = [
    column
    for column in df.columns
    if column not in excluded_columns
]

# Remove high-cardinality free-text or near-identifier string columns.
safe_features = []

for column in candidate_features:
    if df[column].dtype == "object":
        unique_ratio = df[column].nunique(dropna=True) / max(len(df), 1)

        if unique_ratio > 0.50:
            continue

    safe_features.append(column)

if not safe_features:
    raise ValueError("No safe candidate features remained after exclusions.")

model_df = df[[group_col, target_col] + safe_features].copy()

# Keep rows with a usable client and binary target.
model_df = model_df.dropna(subset=[group_col, target_col])
model_df[target_col] = pd.to_numeric(
    model_df[target_col],
    errors="coerce",
)
model_df = model_df.dropna(subset=[target_col])
model_df = model_df[model_df[target_col].isin([0, 1])].copy()
model_df[target_col] = model_df[target_col].astype(int)

X = model_df[safe_features]
y = model_df[target_col]
groups = model_df[group_col]

print(f"\nRows used for validation: {len(model_df):,}")
print(f"Candidate features used: {len(safe_features)}")
print(f"Outcome base rate: {y.mean():.4f}")


# ------------------------------------------------------------
# 4. Build the same Logistic Regression pipeline for both splits
# ------------------------------------------------------------

numeric_features = X.select_dtypes(
    include=["number", "bool"]
).columns.tolist()

categorical_features = [
    column
    for column in safe_features
    if column not in numeric_features
]

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                min_frequency=5,
            ),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ],
    remainder="drop",
)

def make_model():
    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            (
                "model",
                LogisticRegression(
                    max_iter=1000,
                    random_state=42,
                ),
            ),
        ]
    )


# ------------------------------------------------------------
# 5. Evaluation helpers
# ------------------------------------------------------------

def precision_at_k(y_true, probabilities, k=50):
    y_true = np.asarray(y_true)
    probabilities = np.asarray(probabilities)

    k = min(k, len(y_true))

    if k == 0:
        return np.nan

    top_indices = np.argsort(probabilities)[::-1][:k]
    return float(y_true[top_indices].mean())


def recall_at_k(y_true, probabilities, k=50):
    y_true = np.asarray(y_true)
    probabilities = np.asarray(probabilities)

    total_positives = y_true.sum()

    if total_positives == 0:
        return np.nan

    k = min(k, len(y_true))
    top_indices = np.argsort(probabilities)[::-1][:k]

    return float(y_true[top_indices].sum() / total_positives)


def evaluate_split(
    split_name,
    train_indices,
    test_indices,
):
    X_train = X.iloc[train_indices]
    X_test = X.iloc[test_indices]

    y_train = y.iloc[train_indices]
    y_test = y.iloc[test_indices]

    train_groups = set(groups.iloc[train_indices])
    test_groups = set(groups.iloc[test_indices])
    overlap = train_groups.intersection(test_groups)

    model = make_model()
    model.fit(X_train, y_train)

    probabilities = model.predict_proba(X_test)[:, 1]
    predictions = (probabilities >= 0.50).astype(int)

    result = {
        "split": split_name,
        "train_rows": len(train_indices),
        "test_rows": len(test_indices),
        "train_clients": len(train_groups),
        "test_clients": len(test_groups),
        "client_overlap": len(overlap),
        "test_base_rate": y_test.mean(),
        "precision_at_20": precision_at_k(
            y_test,
            probabilities,
            k=20,
        ),
        "precision_at_50": precision_at_k(
            y_test,
            probabilities,
            k=50,
        ),
        "recall_at_50": recall_at_k(
            y_test,
            probabilities,
            k=50,
        ),
        "average_precision": average_precision_score(
            y_test,
            probabilities,
        ),
        "roc_auc": roc_auc_score(
            y_test,
            probabilities,
        ),
        "threshold_precision": precision_score(
            y_test,
            predictions,
            zero_division=0,
        ),
        "threshold_recall": recall_score(
            y_test,
            predictions,
            zero_division=0,
        ),
    }

    return result, model, probabilities, y_test


# ------------------------------------------------------------
# 6. BEFORE — random row split
# ------------------------------------------------------------

all_indices = np.arange(len(model_df))

random_train_indices, random_test_indices = train_test_split(
    all_indices,
    test_size=0.20,
    random_state=42,
    stratify=y,
)


# ------------------------------------------------------------
# 7. AFTER — grouped split by client
# ------------------------------------------------------------

group_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42,
)

group_train_indices, group_test_indices = next(
    group_splitter.split(
        X,
        y,
        groups=groups,
    )
)


# ------------------------------------------------------------
# 8. Train and compare
# ------------------------------------------------------------

random_result, random_model, random_probabilities, random_y_test = (
    evaluate_split(
        split_name="Before: random row split",
        train_indices=random_train_indices,
        test_indices=random_test_indices,
    )
)

grouped_result, grouped_model, grouped_probabilities, grouped_y_test = (
    evaluate_split(
        split_name="After: client-grouped split",
        train_indices=group_train_indices,
        test_indices=group_test_indices,
    )
)

comparison = pd.DataFrame(
    [random_result, grouped_result]
)

display_columns = [
    "split",
    "train_rows",
    "test_rows",
    "train_clients",
    "test_clients",
    "client_overlap",
    "test_base_rate",
    "precision_at_20",
    "precision_at_50",
    "recall_at_50",
    "average_precision",
    "roc_auc",
]

comparison_display = comparison[display_columns].copy()

metric_columns = [
    "test_base_rate",
    "precision_at_20",
    "precision_at_50",
    "recall_at_50",
    "average_precision",
    "roc_auc",
]

comparison_display[metric_columns] = (
    comparison_display[metric_columns].round(4)
)

print("\nBefore/after validation comparison")
display(comparison_display)


# ------------------------------------------------------------
# 9. Explicit validation checks
# ------------------------------------------------------------

random_overlap = comparison.loc[
    comparison["split"] == "Before: random row split",
    "client_overlap",
].iloc[0]

grouped_overlap = comparison.loc[
    comparison["split"] == "After: client-grouped split",
    "client_overlap",
].iloc[0]

assert grouped_overlap == 0, (
    "Grouped validation failed: some clients occur in both sets."
)

print("Validation checks")
print("- Random-split client overlap:", random_overlap)
print("- Grouped-split client overlap:", grouped_overlap)
print("- Grouped split passed the zero-client-overlap check.")


# Save objects for the later error-analysis section.
audit_artifacts = {
    "model_df": model_df,
    "features": safe_features,
    "target_col": target_col,
    "group_col": group_col,
    "group_train_indices": group_train_indices,
    "group_test_indices": group_test_indices,
    "grouped_model": grouped_model,
    "grouped_probabilities": grouped_probabilities,
    "grouped_y_test": grouped_y_test,
}

print("\nSection 2 completed successfully.")


Created target: is_declining_label
is_declining_label
1    16262
0    13738
Name: count, dtype: int64
Overall decline rate: 0.5421
Dataset path: /content/flyrank-ml/data/raw/content_refresh_anonymized.csv
Rows: 30,000
Columns: 45

Target column: is_declining_label
Grouping column: client_id

Rows used for validation: 30,000
Candidate features used: 40
Outcome base rate: 0.5421

Before/after validation comparison


,split,train_rows,test_rows,train_clients,test_clients,client_overlap,test_base_rate,precision_at_20,precision_at_50,recall_at_50,average_precision,roc_auc
0,Before: random row split,24000,6000,32,31,31,0.542,1.0,1.0,0.0154,0.9375,0.9207
1,After: client-grouped split,23837,6163,25,7,0,0.511,1.0,1.0,0.0159,0.8507,0.8378


Validation checks
- Random-split client overlap: 31
- Grouped-split client overlap: 0
- Grouped split passed the zero-client-overlap check.

Section 2 completed successfully.


### Interpretation of the before/after comparison

The random row split measured a ROC-AUC of 0.9207, while the client-grouped split measured a lower ROC-AUC of 0.8378. Average precision also decreased from 0.9375 to 0.8507.

The random split contained 31 clients in both the training and test sets. Therefore, its result may partly reflect patterns learned from clients that were already represented during training. The grouped split had zero client overlap and provides a more honest estimate for performance on the seven held-out clients in this dataset.

However, both splits measured Precision@20 and Precision@50 of 1.00. These unusually strong ranking results require further investigation rather than being presented as evidence of production-level performance. In particular, the target was created from `trend_direction`, while some candidate features contain current-versus-previous 30-day measurements that may also have been used to calculate that trend.

For that reason, the results above are treated as an intermediate diagnostic. The next leakage audit checks whether the model is indirectly receiving information used to construct the label. No causal or business-impact claim is made from this comparison.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [13]:
# ============================================================
# 3. Leakage audit
# Check whether features contain information used to create
# the target and re-run the grouped model with safer features.
# ============================================================

# The target is derived from trend_direction == "down".
# Current-versus-previous period columns may have been used
# to calculate that trend, so they are considered label-construction
# proxies and excluded from the safer model.

label_source_columns = [
    "trend_direction",
    "trend_pct",
]

period_comparison_columns = [
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
]

direct_identifiers = [
    "content_id",
    "client_id",
]

audit_rows = []

for column in df.columns:
    if column == target_col:
        category = "target"
        decision = "exclude"
        reason = "This is the prediction label."

    elif column in label_source_columns:
        category = "direct label source"
        decision = "exclude"
        reason = (
            "This column directly defines or quantifies "
            "the target trend."
        )

    elif column in period_comparison_columns:
        category = "label-construction proxy"
        decision = "exclude"
        reason = (
            "Current-versus-previous period values may be used "
            "to calculate trend_direction."
        )

    elif column in direct_identifiers:
        category = "identifier/grouping field"
        decision = "exclude"
        reason = (
            "Used for identification or grouped validation, "
            "not as a predictive feature."
        )

    else:
        category = "candidate predictor"
        decision = "retain for this audit"
        reason = (
            "No direct label-construction relationship "
            "identified from the column definition."
        )

    audit_rows.append(
        {
            "column": column,
            "category": category,
            "decision": decision,
            "reason": reason,
        }
    )

leakage_audit = pd.DataFrame(audit_rows)

print("Columns excluded because of leakage or identification risk:")
display(
    leakage_audit[
        leakage_audit["decision"] == "exclude"
    ].reset_index(drop=True)
)


# ------------------------------------------------------------
# Create a safer feature set
# ------------------------------------------------------------

safer_exclusions = set(
    label_source_columns
    + period_comparison_columns
    + direct_identifiers
    + [
        target_col,
        "decline_proxy",
        "is_declining",
        "reason_code",
        "action_label",
        "action_score",
        "rank",
    ]
)

safer_features = [
    column
    for column in df.columns
    if column not in safer_exclusions
]

# Remove high-cardinality text columns that may act like identifiers.
final_safe_features = []

for column in safer_features:
    if df[column].dtype == "object":
        unique_ratio = (
            df[column].nunique(dropna=True) / max(len(df), 1)
        )

        if unique_ratio > 0.50:
            continue

    final_safe_features.append(column)

print(f"\nOriginal candidate feature count: {len(safe_features)}")
print(f"Safer feature count: {len(final_safe_features)}")
print("\nRemoved period-comparison features:")

for column in period_comparison_columns:
    print("-", column)


# ------------------------------------------------------------
# Prepare grouped train/test data using the same client split
# ------------------------------------------------------------

safe_X = model_df[final_safe_features].copy()
safe_y = model_df[target_col].copy()
safe_groups = model_df[group_col].copy()

safe_numeric_features = safe_X.select_dtypes(
    include=["number", "bool"]
).columns.tolist()

safe_categorical_features = [
    column
    for column in final_safe_features
    if column not in safe_numeric_features
]

safe_numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

safe_categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                min_frequency=5,
            ),
        ),
    ]
)

safe_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            safe_numeric_pipeline,
            safe_numeric_features,
        ),
        (
            "categorical",
            safe_categorical_pipeline,
            safe_categorical_features,
        ),
    ],
    remainder="drop",
)

safe_model = Pipeline(
    steps=[
        ("preprocessor", safe_preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                random_state=42,
            ),
        ),
    ]
)

X_train_safe = safe_X.iloc[group_train_indices]
X_test_safe = safe_X.iloc[group_test_indices]

y_train_safe = safe_y.iloc[group_train_indices]
y_test_safe = safe_y.iloc[group_test_indices]

safe_model.fit(X_train_safe, y_train_safe)

safe_probabilities = safe_model.predict_proba(
    X_test_safe
)[:, 1]

safe_predictions = (
    safe_probabilities >= 0.50
).astype(int)


# ------------------------------------------------------------
# Compare grouped result before and after leakage exclusions
# ------------------------------------------------------------

safe_result = {
    "model_version": (
        "After leakage audit: grouped split + safer features"
    ),
    "test_rows": len(y_test_safe),
    "test_clients": safe_groups.iloc[
        group_test_indices
    ].nunique(),
    "client_overlap": 0,
    "test_base_rate": y_test_safe.mean(),
    "precision_at_20": precision_at_k(
        y_test_safe,
        safe_probabilities,
        20,
    ),
    "precision_at_50": precision_at_k(
        y_test_safe,
        safe_probabilities,
        50,
    ),
    "recall_at_50": recall_at_k(
        y_test_safe,
        safe_probabilities,
        50,
    ),
    "average_precision": average_precision_score(
        y_test_safe,
        safe_probabilities,
    ),
    "roc_auc": roc_auc_score(
        y_test_safe,
        safe_probabilities,
    ),
}

pre_audit_grouped_result = {
    "model_version": (
        "Before leakage audit: grouped split + proxy features"
    ),
    "test_rows": grouped_result["test_rows"],
    "test_clients": grouped_result["test_clients"],
    "client_overlap": grouped_result["client_overlap"],
    "test_base_rate": grouped_result["test_base_rate"],
    "precision_at_20": grouped_result["precision_at_20"],
    "precision_at_50": grouped_result["precision_at_50"],
    "recall_at_50": grouped_result["recall_at_50"],
    "average_precision": grouped_result["average_precision"],
    "roc_auc": grouped_result["roc_auc"],
}

leakage_comparison = pd.DataFrame(
    [
        pre_audit_grouped_result,
        safe_result,
    ]
)

leakage_metric_columns = [
    "test_base_rate",
    "precision_at_20",
    "precision_at_50",
    "recall_at_50",
    "average_precision",
    "roc_auc",
]

leakage_comparison[leakage_metric_columns] = (
    leakage_comparison[leakage_metric_columns].round(4)
)

print("\nGrouped validation before/after leakage exclusions")
display(leakage_comparison)


# ------------------------------------------------------------
# Real failure examples from the safer grouped model
# ------------------------------------------------------------

# ------------------------------------------------------------
# Real failure examples from the safer grouped model
# ------------------------------------------------------------

# Locate the original dataframe rows corresponding to the
# grouped test set. content_id was intentionally excluded from
# model_df, so it is retrieved from the original dataframe only
# for human-readable error inspection.

test_model_rows = model_df.iloc[group_test_indices].copy()
test_original_indices = test_model_rows.index

error_examples = df.loc[
    test_original_indices,
    [
        group_col,
        "content_id",
    ],
].copy()

# Preserve exactly the same row order as the prediction arrays.
error_examples = error_examples.loc[test_original_indices].copy()

error_examples[target_col] = (
    test_model_rows[target_col].to_numpy()
)

error_examples["predicted_probability"] = safe_probabilities
error_examples["predicted_label"] = safe_predictions

error_examples["error_type"] = np.select(
    [
        (
            (error_examples[target_col] == 0)
            & (error_examples["predicted_label"] == 1)
        ),
        (
            (error_examples[target_col] == 1)
            & (error_examples["predicted_label"] == 0)
        ),
    ],
    [
        "false_positive",
        "false_negative",
    ],
    default="correct",
)

false_positives = (
    error_examples[
        error_examples["error_type"] == "false_positive"
    ]
    .sort_values(
        "predicted_probability",
        ascending=False,
    )
    .head(5)
)

false_negatives = (
    error_examples[
        error_examples["error_type"] == "false_negative"
    ]
    .sort_values(
        "predicted_probability",
        ascending=True,
    )
    .head(5)
)

print("\nHigh-confidence false positives")
display(false_positives)

print("\nHigh-confidence false negatives")
display(false_negatives)


# Save safer artifacts for later sections.
audit_artifacts.update(
    {
        "final_safe_features": final_safe_features,
        "safe_model": safe_model,
        "safe_probabilities": safe_probabilities,
        "safe_predictions": safe_predictions,
        "y_test_safe": y_test_safe,
        "error_examples": error_examples,
        "leakage_comparison": leakage_comparison,
    }
)

print("\nSection 3 completed successfully.")

# Save safer artifacts for later sections.
audit_artifacts.update(
    {
        "final_safe_features": final_safe_features,
        "safe_model": safe_model,
        "safe_probabilities": safe_probabilities,
        "safe_predictions": safe_predictions,
        "y_test_safe": y_test_safe,
        "error_examples": error_examples,
        "leakage_comparison": leakage_comparison,
    }
)

print("\nSection 3 completed successfully.")

Columns excluded because of leakage or identification risk:


,column,category,decision,reason
0,content_id,identifier/grouping field,exclude,"Used for identification or grouped validation,..."
1,client_id,identifier/grouping field,exclude,"Used for identification or grouped validation,..."
2,impressions_last_30d,label-construction proxy,exclude,Current-versus-previous period values may be u...
3,clicks_last_30d,label-construction proxy,exclude,Current-versus-previous period values may be u...
4,sessions_last_30d,label-construction proxy,exclude,Current-versus-previous period values may be u...
5,impressions_prev_30d,label-construction proxy,exclude,Current-versus-previous period values may be u...
6,clicks_prev_30d,label-construction proxy,exclude,Current-versus-previous period values may be u...
7,sessions_prev_30d,label-construction proxy,exclude,Current-versus-previous period values may be u...
8,trend_direction,direct label source,exclude,This column directly defines or quantifies the...
9,trend_pct,direct label source,exclude,This column directly defines or quantifies the...



Original candidate feature count: 40
Safer feature count: 34

Removed period-comparison features:
- impressions_last_30d
- clicks_last_30d
- sessions_last_30d
- impressions_prev_30d
- clicks_prev_30d
- sessions_prev_30d

Grouped validation before/after leakage exclusions


,model_version,test_rows,test_clients,client_overlap,test_base_rate,precision_at_20,precision_at_50,recall_at_50,average_precision,roc_auc
0,Before leakage audit: grouped split + proxy fe...,6163,7,0,0.511,1.00,1.00,0.0159,0.8507,0.8378
1,After leakage audit: grouped split + safer fea...,6163,7,0,0.511,0.65,0.64,0.0102,0.5686,0.5771



High-confidence false positives


,client_id,content_id,is_declining_label,predicted_probability,predicted_label,error_type
20736,client_8527a891e2,content_41baf0722ad9,0,0.912115,1,false_positive
10175,client_f369cb89fc,content_374e795aab68,0,0.906410,1,false_positive
18531,client_4e07408562,content_d10f9ce1e0cd,0,0.898958,1,false_positive
11887,client_8527a891e2,content_ce59581533ca,0,0.898853,1,false_positive
26614,client_f369cb89fc,content_7be5f150dc65,0,0.897585,1,false_positive



High-confidence false negatives


,client_id,content_id,is_declining_label,predicted_probability,predicted_label,error_type
4081,client_e629fa6598,content_917fc1b11fe1,1,0.048786,0,false_negative
12845,client_e629fa6598,content_742a8fcba2fe,1,0.050433,0,false_negative
28032,client_e629fa6598,content_b2beaf2fc81c,1,0.070214,0,false_negative
10486,client_e629fa6598,content_7a82fa0b634e,1,0.072606,0,false_negative
6824,client_e629fa6598,content_d987db4e38fe,1,0.073318,0,false_negative



Section 3 completed successfully.

Section 3 completed successfully.


### Leakage audit interpretation

The initial grouped validation still produced unusually strong ranking results, including Precision@50 of 1.00 and ROC-AUC of 0.8378. The feature audit found that the model included current and previous 30-day measurements that may have been used to construct `trend_direction`, which is the source of the prediction label.

After removing these possible label-construction proxies, grouped ROC-AUC decreased from 0.8378 to 0.5771. Average precision decreased from 0.8507 to 0.5686, and Precision@50 decreased from 1.00 to 0.64.

This reduction suggests that a substantial part of the earlier measured performance was associated with features closely connected to label construction. The safer result is less impressive, but it is more appropriate for describing directional performance on the seven held-out clients.

The post-audit model measured Precision@50 of 0.64 compared with a test base rate of 0.511. This is an observed offline result and does not establish causal business impact or production performance. The model should be treated as decision-support for human review.

## 4. Claim rewrite

### Claim 1

**Earlier claim:**

The model accurately identifies content that will decline.

**Rewritten public-safe claim:**

In an offline client-grouped evaluation, the leakage-audited Logistic Regression model measured a ROC-AUC of 0.5771 on seven held-out clients. This provides limited directional evidence that the available features contain some signal associated with the observed decline label.

The result does not show that the model can reliably predict future decline in production.

---

### Claim 2

**Earlier claim:**

The machine learning model performs better than the existing baseline.

**Rewritten public-safe claim:**

For the first 50 ranked records in this test set, the leakage-audited model measured Precision@50 of 0.64, compared with a test outcome base rate of 0.511.

This is an observed offline result from one grouped split. It does not establish that the model is consistently better than the fixed-rule baseline across clients, time periods, or production settings.

---

### Claim 3

**Earlier claim:**

The model can improve business performance by finding declining content.

**Rewritten public-safe claim:**

The model may support human review by ranking records that appear more likely to have the observed decline label.

The evaluation did not measure revenue, traffic recovery, editorial improvement, or causal business impact. A production test would be required before making a business-performance claim.

---

### Claim 4

**Earlier claim:**

The model achieved very strong prediction performance.

**Rewritten public-safe claim:**

Before the leakage audit, the grouped model measured ROC-AUC of 0.8378 and Precision@50 of 1.00. After removing features that may have contributed to label construction, ROC-AUC decreased to 0.5771 and Precision@50 decreased to 0.64.

The post-audit result is the more appropriate estimate to report. The difference demonstrates why feature provenance and label construction must be reviewed before model performance is communicated.

---

### Final supported claim

In this offline experiment, a Logistic Regression model was evaluated using a client-grouped split with zero client overlap. After removing features closely connected to label construction, the model measured ROC-AUC of 0.5771, average precision of 0.5686, and Precision@50 of 0.64 on seven held-out clients.

These results are directional and specific to this dataset and split. The output is intended as decision-support for human review and does not establish causal impact, production reliability, or future business improvement.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

### Remaining limitations

The target is a proxy label derived from `trend_direction`, not a directly measured business outcome.

The evaluation uses one client-grouped split with seven held-out clients. Results may vary under another client split or time period.

The exact upstream formula used to create `trend_direction` was not independently verified. Current-versus-previous 30-day fields were excluded conservatively because they may contribute to label construction.

The model was evaluated offline. It was not deployed, and no A/B test or prospective production evaluation was conducted.

### Audit conclusion

The audit changed the interpretation of the Week-5 model. After removing features closely connected to label construction, the measured performance decreased substantially.

The post-audit model retained modest directional performance and should be treated as decision-support for human review, not as evidence of causal impact or guaranteed production performance.